# 台积电季度指引原始 PDF 传输预检

冻结三个跨期官方附件，只验证原件、文本、时点字段和 NAS 备份。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
import fitz
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_tsmc_guidance_pdf_transport_20261023';raw=root/'data/snapshots/stage1_tsmc_guidance_pdf_transport_20261023';raw.mkdir(parents=True,exist_ok=True);plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and len(plan['exact_requests'])==3
state={'status':'started','GET_attempts':0,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];checks=[];failure=None
for i,spec in enumerate(plan['exact_requests'],1):
 body=raw/f'anchor_{i}.pdf';headers=raw/f'anchor_{i}.headers';state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','45','--max-filesize','8000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A','Mozilla/5.0 soxl-qlib-source-research','-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=60);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
  receipt={'anchor':spec['anchor'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'anchor':spec['anchor'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'exception':str(exc)[:300]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None)
 receipts.append(receipt);(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.read_bytes()[:4]!=b'%PDF':failure=f'anchor_{i}_transport_or_pdf_magic';break
 try:
  pdf=fitz.open(body);txt='\n'.join(page.get_text() for page in pdf[:min(3,len(pdf))]);assert len(txt)>300
  low=txt.lower();assert 'tsmc' in low and ('quarter' in low or 'q22' in low or 'q26' in low) and ('revenue' in low or 'gross margin' in low)
  checks.append({'anchor':spec['anchor'],'raw_sha256':sha(body),'page_count':len(pdf),'first_three_page_text_chars':len(txt),'contains_issuer':True,'contains_quarter_and_financial_fields':True,'first_three_page_excerpt':txt[:350]})
  (card/'source_checks.json').write_text(json.dumps(checks,indent=2)+'\n')
 except Exception as exc:failure=f'anchor_{i}_semantic_{type(exc).__name__}_{str(exc)[:120]}';break
 time.sleep(plan['request']['seconds_between_requests'])
files=[{'name':x.name,'sha256':sha(x),'bytes':x.stat().st_size} for x in sorted(raw.iterdir()) if x.is_file()];manifest={'plan_sha256':sha(card/'plan.json'),'files':files,'receipts_sha256':sha(card/'capture_receipts.json')};(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='tsmc-guidance-pdf-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ('plan.json','capture_receipts.json','source_manifest.json'):t.add(card/name,arcname=name)
 digest=sha(archive);a=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert a['sha256']==digest and a['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='tsmc-guidance-pdf-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in files:
    m=t.extractfile('raw/'+x['name']);assert m and hashlib.sha256(m.read()).hexdigest()==x['sha256']
result={'status':'official_PDF_transport_three_anchors_passed_first_public_pending' if not failure and len(checks)==3 else 'bounded_official_PDF_transport_partial_or_failed','failure':failure,'plan_sha256':sha(card/'plan.json'),'GET_attempts':state['GET_attempts'],'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'historical_first_public_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,indent=2)+'\n');state.update(status='completed' if not failure else 'partial_or_failed',NAS_archive_sha256=digest);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'checks':len(checks),'NAS':digest})


{'status': 'bounded_official_PDF_transport_partial_or_failed', 'failure': 'anchor_1_transport_or_pdf_magic', 'GETs': 1, 'checks': 0, 'NAS': 'b1d818f59d311a6b8ced777249f8bb8d0b092fe44a9384afa8dc5a6a86e8f88c'}


## 中文结论

首个官方 PDF URL 返回 HTTP403，按冻结计划停止，另外两次请求未执行。失败响应精确字节已备份 NAS；零特征、拟合或账户。见 `conclusion.md`。